In [1]:
# Instalação e configuração (Google Colab)
!apt-get update > /dev/null
!apt install chromium-chromedriver -y > /dev/null
!cp /usr/lib/chromium-browser/chromedriver /usr/bin
!pip install selenium beautifulsoup4 > /dev/null

# Imports
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, ElementClickInterceptedException
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup

# Configurações do navegador
options = Options()
options.binary_location = "/usr/bin/chromium-browser"
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

# Inicializa o navegador
driver = webdriver.Chrome(options=options)

# URL da categoria de porcelanatos
url = "https://www.telhanorte.com.br/pisos-e-revestimentos/porcelanatos"
driver.get(url)
time.sleep(5)

# Clicar no botão "Mostrar mais" até que ele desapareça
while True:
    try:
        botao_mais = driver.find_element(By.CSS_SELECTOR, "a.vtex-button[rel='next']")
        driver.execute_script("arguments[0].scrollIntoView(true);", botao_mais)
        time.sleep(1)
        botao_mais.click()
        time.sleep(3)
    except (NoSuchElementException, ElementClickInterceptedException):
        print("Não há mais páginas para carregar.")
        break

# Coleta os elementos de produto
produtos_elementos = driver.find_elements(By.CSS_SELECTOR, "div.vtex-search-result-3-x-galleryItem")

# Extrair links
links = []
for produto in produtos_elementos:
    try:
        link = produto.find_element(By.CSS_SELECTOR, "a").get_attribute("href")
        if link:
            links.append(link)
    except:
        continue

driver.quit()  # Fecha o navegador

# Função para extrair detalhes de um produto
def extrair_detalhes_produto(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    }
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        return None

    soup = BeautifulSoup(response.text, "html.parser")

    try:
        nome_produto = soup.find("h1", class_="vtex-store-components-3-x-productNameContainer").text.strip()
    except:
        nome_produto = "N/A"

    codigo_tag = soup.find("span", class_="vtex-product-identifier-0-x-product-identifier__value")
    codigo_produto = codigo_tag.text.strip() if codigo_tag else "0"

    preco_original_tag = soup.select_one(".vtex-product-price-1-x-listPriceWithUnitMultiplier strike")
    preco_original = preco_original_tag.text.strip() if preco_original_tag else "N/A"

    preco_promocional_tag = soup.select_one(".telhanorte-telha-store-app-1-x-preco-best-un")
    preco_promocional = preco_promocional_tag.text.strip() if preco_promocional_tag else "N/A"

    preco_m2_tag = soup.select_one(".telhanorte-telha-store-app-1-x-preco-m2")
    preco_m2 = preco_m2_tag.text.strip() if preco_m2_tag else "N/A"

    imagem_tag = soup.find("img", class_="telhanorte-telha-store-app-1-x-imagem-destaque")
    imagem_url = imagem_tag['src'] if imagem_tag else "N/A"

    return [nome_produto, codigo_produto, preco_original, preco_promocional, preco_m2, imagem_url, url]

# Coletar os detalhes de cada produto
dados_produtos = []
for i, link in enumerate(links):
    print(f"Processando ({i+1}/{len(links)}): {link}")
    detalhes = extrair_detalhes_produto(link)
    if detalhes:
        dados_produtos.append(detalhes)

# Salvar em CSV
df = pd.DataFrame(dados_produtos, columns=[
    "Nome", "Código", "Preço Original", "Preço Promocional", "Preço por m²", "Imagem", "URL"
])
df.to_csv("telhanorte_porcelanatos_detalhado.csv", index=False)
df.head()


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


cp: '/usr/lib/chromium-browser/chromedriver' and '/usr/bin/chromedriver' are the same file
Não há mais páginas para carregar.
Processando (1/40): https://www.telhanorte.com.br/porcelanato-nord-ris-90x90-nat-ret-cx-2-42-m2-portobello-2547910/p
Processando (2/40): https://www.telhanorte.com.br/porcelanato-confete-white-natural-retificado-100x100cm-caixa-c-2m2-ceusa-1751271/p
Processando (3/40): https://www.telhanorte.com.br/porcelanato-mont-blanc-acetinado-90x90cm-ret-cx1-62m2-eliane-2518449/p
Processando (4/40): https://www.telhanorte.com.br/porcelanato-esmaltado-60x60-cm-caixa-com-2-2-m2-cinza-cemento-grigio-biancogres-2363550/p
Processando (5/40): https://www.telhanorte.com.br/porcelanato-caminho-sgr-natural-877x877cm-retificado-caixa-154m2-portinari-2515032/p
Processando (6/40): https://www.telha

,Nome,Código,Preço Original,Preço Promocional,Preço por m²,Imagem,URL
0,Porcelanato Nord Ris 90X90Cm Natural Retificad...,2547910,N/A,"R$ 250,00","R$ 103,31/m²",https://telhanorte.vtexassets.com/arquivos/ids...,https://www.telhanorte.com.br/porcelanato-nord...
1,Porcelanato Confete White Natural 100x100cm Re...,1751271,N/A,"R$ 300,61","R$ 150,31/m²",https://telhanorte.vtexassets.com/arquivos/ids...,https://www.telhanorte.com.br/porcelanato-conf...
2,Porcelanato Mont Blanc Acetinado 90X90Cm Ret C...,2518449,N/A,"R$ 181,28","R$ 111,90/m²",https://telhanorte.vtexassets.com/arquivos/ids...,https://www.telhanorte.com.br/porcelanato-mont...
3,Porcelanato Cimentício Acetinado Borda Reta In...,2363550,N/A,"R$ 130,08","R$ 59,13/m²",https://telhanorte.vtexassets.com/arquivos/ids...,https://www.telhanorte.com.br/porcelanato-esma...
4,"Porcelanato Caminho Sgr Natural 87,7X87,7Cm Re...",2515032,N/A,"R$ 246,25","R$ 159,90/m²",https://telhanorte.vtexassets.com/arquivos/ids...,https://www.telhanorte.com.br/porcelanato-cami...
